In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
import os
import json
from typing import Literal


In [ ]:
def plot_frames_vs_proprio(df, embodiment, stat, proprio_type: Literal["cartesian", "joints"]):
    if proprio_type == "cartesian":
        key = "ee_pose"
    elif proprio_type == "joints":
        key = "joint_positions"
    d = df[(df["embodiment"] == embodiment) & (df["key"] == key)].copy()
    d = d.sort_values("frames").reset_index(drop=True)

    frames = d["frames"].to_numpy()
    stat_paths = d[f"{stat}_path"].values
    print(stat_paths)

    vecs = []
    for p in stat_paths:
        x = torch.load(p, map_location="cpu")
        if isinstance(x, torch.Tensor):
            t = x.detach().cpu().float()
        else:
            t = torch.as_tensor(x).detach().cpu().float()
        print("t")
        print(t)
        vecs.append(t.numpy())

    Y = np.stack(vecs, axis=0)
    D = Y.shape[-1]
    print(Y.shape)

    for i in range(D):
        plt.figure(figsize=(10, 5))
        plt.plot(frames, Y[:, i], label=f"{stat}[{i}]")
        plt.xlabel("frames")
        plt.ylabel("eepose")
        plt.title(f"{embodiment} eepose vs frames: {stat}[{i}]")
        plt.legend()
        plt.show()

In [ ]:
def plot_frames_vs_action(df, embodiment, stat, seq_i, action_type: Literal["cartesian", "joints"]):
    d = df[(df["embodiment"] == embodiment) & (df["key"] == f"actions_{action_type}")].copy()
    d = d.sort_values("frames").reset_index(drop=True)

    frames = d["frames"].to_numpy()
    stat_paths = d[f"{stat}_path"].values
    print

    vecs = []
    for p in stat_paths:
        x = torch.load(p, map_location="cpu")
        if isinstance(x, torch.Tensor):
            t = x.detach().cpu().float()
        else:
            t = torch.as_tensor(x).detach().cpu().float()
        print("t shape")
        print(t.shape)
        vecs.append(t.numpy())

    Y = np.stack(vecs, axis=0)
    Y = Y[:,seq_i,:]
    D = Y.shape[-1]

    for i in range(D):
        plt.figure(figsize=(10, 5))
        plt.plot(frames, Y[:, i], label=f"{stat}[{i}]")
        plt.xlabel("frames")
        plt.ylabel("eepose")
        plt.title(f"{embodiment} eepose vs frames: {stat}[{i}]")
        plt.legend()
        plt.show()

In [ ]:
def plot_frames_vs_time(df):
    d = df.copy()
    d["total_time"] = d["loading_time"] + d["computing_time"]

    d = d[["frames", "computing_time", "loading_time", "total_time"]].drop_duplicates()

    d = d.sort_values("frames").reset_index(drop=True)

    frames = d["frames"].to_numpy()
    computing_time = d["computing_time"].to_numpy()
    loading_time = d["loading_time"].to_numpy()
    total_time = d["total_time"].to_numpy()

    fig, axs = plt.subplots(3, 1, figsize=(10, 15), sharex=True)
    axs[0].plot(frames, computing_time, label="computing_time")
    axs[1].plot(frames, loading_time, label="loading_time")
    axs[2].plot(frames, total_time, label="total_time")

    for ax in axs:
        ax.set_xlabel("frames")
        ax.set_ylabel("time(s)")
        ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
csv_path = "/coc/flash7/paphiwetsa3/projects/EgoVerse/logs/test/test_2026-02-27_13-34-33/benchmark_stats.csv"
df = pd.read_csv(csv_path)
df

In [ ]:
plot_frames_vs_time(df)

In [ ]:
plot_frames_vs_proprio(df, 8, "quantile_1", "cartesian")

In [ ]:
plot_frames_vs_action(df, 8, "mean", 0, "cartesian")